# LSTM Notebook

This notebook provides an example of working with the LSTM model. It
shows the process of loading per-game data for premier league forwards for
the current season, splitting into train/test sets, creating datasets and
dataloaders from them, training a `FootballLSTM` instance, and evaluating
it.

In [1]:
import sys
import os
import pandas as pd
from understatapi import UnderstatClient
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from matplotlib import pyplot as plt
import numpy as np

# Also import local helpers
target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)
    
from preprocess import get_position_players_stats_df, CustomFootballDataset, DifferencedFootballDataset, merge_stats_df_with_transfermarkt
from football_lstm import FootballLSTM, DifferencingFootballLSTM
from utils import hyperparam_tuning

In [2]:
# have GPU available to speed up
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cpu


In [3]:
# Create understat connection
understat = UnderstatClient()

In [ ]:
# Choose what stats to forecast, and how many past games to use to predict
# the current one
stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
per_90_stats = list(map(lambda s: f"{s}_per_90", stats))
games_per_block = 10
differencing = True

In [5]:
### ONLY UNCOMMENT IF NEED TO PULL NEW POSITION DATA ###
### MAKE SURE THAT YOU CHANGE DF NAME ###

# Get forward stats for all seasons in top 5 leagues
# leagues = ["EPL", "La_Liga", "Bundesliga", "Serie_A", "Ligue_1"]
# seasons = list(range(2014,2026))
positions = ['F'] # forwards saved to csv, no need to run again
# 
# stats_df = get_position_players_stats_df(understat, positions, games_per_block, stats, leagues=leagues, seasons=seasons)
#stats_df.head()
# 
# # save dataset to csv to avoid pulls from understat every time as it can take 5-10min for just one position
# os.makedirs("data", exist_ok=True)  
# 
# # to be able to save datasets for a set of positions
# stats_df.to_csv(f"../data/{'_'.join(positions)}_stats_2.csv", index=True)

In [6]:
# Read in saved df
stats_df = pd.read_csv(f"../data/{'_'.join(positions)}_stats_2.csv")

# Convert date to appropriate type
stats_df['date'] = pd.to_datetime(stats_df['date'])

stats_df = stats_df.set_index(['player_id', 'player_name', 'date', 'league'])

stats_df.head()

### MAKE SURE THAT YOU CHANGE DF NAME IF NEW CSV ###
### AND THAT ALL DF BELOW ARE UPDATED TO MATCH NEW DF NAME ###

goals_per_90  \
player_id player_name      date                league                     
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga      0.000000   
                           2015-11-21 18:30:00 Bundesliga      0.281250   
                           2016-02-07 18:30:00 Bundesliga      0.000000   
                           2016-03-05 18:30:00 Bundesliga      0.202247   
                           2016-04-17 19:30:00 Bundesliga      0.000000   

                                                           xG_per_90  \
player_id player_name      date                league                  
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga   0.167702   
                           2015-11-21 18:30:00 Bundesliga   0.043876   
                           2016-02-07 18:30:00 Bundesliga   0.095472   
                           2016-03-05 18:30:00 Bundesliga   0.188908   
                           2016-04-17 19:30:00 Bundesliga   0.003078   

                                                           assists_per_90  \
player_id player_name      date                league                       
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga        0.000000   
                           2015-11-21 18:30:00 Bundesliga        0.000000   
                           2016-02-07 18:30:00 Bundesliga        0.000000   
                           2016-03-05 18:30:00 Bundesliga        0.202247   
                           2016-04-17 19:30:00 Bundesliga        0.430622   

                                                           xA_per_90  \
player_id player_name      date                league                  
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga   0.000000   
                           2015-11-21 18:30:00 Bundesliga   0.206763   
                           2016-02-07 18:30:00 Bundesliga   0.068278   
                           2016-03-05 18:30:00 Bundesliga   0.121450   
                           2016-04-17 19:30:00 Bundesliga   0.155947   

                                                           key_passes_per_90  \
player_id player_name      date                league                          
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga           0.000000   
                           2015-11-21 18:30:00 Bundesliga           1.968750   
                           2016-02-07 18:30:00 Bundesliga           1.253482   
                           2016-03-05 18:30:00 Bundesliga           1.011236   
                           2016-04-17 19:30:00 Bundesliga           0.861244   

                                                           xGChain_per_90  \
player_id player_name      date                league                       
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga        0.239728   
                           2015-11-21 18:30:00 Bundesliga        0.289482   
                           2016-02-07 18:30:00 Bundesliga        0.303476   
                           2016-03-05 18:30:00 Bundesliga        0.513151   
                           2016-04-17 19:30:00 Bundesliga        0.401483   

                                                           xGBuildup_per_90  
player_id player_name      date                league                        
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga          0.110539  
                           2015-11-21 18:30:00 Bundesliga          0.185229  
                           2016-02-07 18:30:00 Bundesliga          0.181097  
                           2016-03-05 18:30:00 Bundesliga          0.224042  
                           2016-04-17 19:30:00 Bundesliga          0.245536

In [7]:
# Train/test split by player
all_players = stats_df.index.get_level_values('player_id').unique()

# 80/20 split
np.random.seed(42)
train_players = np.random.choice(all_players, size=int(0.8*len(all_players)), replace=False)

train_df = stats_df[stats_df.index.get_level_values('player_id').isin(train_players)]
test_df = stats_df[~stats_df.index.get_level_values('player_id').isin(train_players)]

In [8]:
train_df.head()

goals_per_90  \
player_id player_name      date                league                     
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga      0.000000   
                           2015-11-21 18:30:00 Bundesliga      0.281250   
                           2016-02-07 18:30:00 Bundesliga      0.000000   
                           2016-03-05 18:30:00 Bundesliga      0.202247   
                           2016-04-17 19:30:00 Bundesliga      0.000000   

                                                           xG_per_90  \
player_id player_name      date                league                  
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga   0.167702   
                           2015-11-21 18:30:00 Bundesliga   0.043876   
                           2016-02-07 18:30:00 Bundesliga   0.095472   
                           2016-03-05 18:30:00 Bundesliga   0.188908   
                           2016-04-17 19:30:00 Bundesliga   0.003078   

                                                           assists_per_90  \
player_id player_name      date                league                       
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga        0.000000   
                           2015-11-21 18:30:00 Bundesliga        0.000000   
                           2016-02-07 18:30:00 Bundesliga        0.000000   
                           2016-03-05 18:30:00 Bundesliga        0.202247   
                           2016-04-17 19:30:00 Bundesliga        0.430622   

                                                           xA_per_90  \
player_id player_name      date                league                  
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga   0.000000   
                           2015-11-21 18:30:00 Bundesliga   0.206763   
                           2016-02-07 18:30:00 Bundesliga   0.068278   
                           2016-03-05 18:30:00 Bundesliga   0.121450   
                           2016-04-17 19:30:00 Bundesliga   0.155947   

                                                           key_passes_per_90  \
player_id player_name      date                league                          
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga           0.000000   
                           2015-11-21 18:30:00 Bundesliga           1.968750   
                           2016-02-07 18:30:00 Bundesliga           1.253482   
                           2016-03-05 18:30:00 Bundesliga           1.011236   
                           2016-04-17 19:30:00 Bundesliga           0.861244   

                                                           xGChain_per_90  \
player_id player_name      date                league                       
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga        0.239728   
                           2015-11-21 18:30:00 Bundesliga        0.289482   
                           2016-02-07 18:30:00 Bundesliga        0.303476   
                           2016-03-05 18:30:00 Bundesliga        0.513151   
                           2016-04-17 19:30:00 Bundesliga        0.401483   

                                                           xGBuildup_per_90  
player_id player_name      date                league                        
140       Yannick Gerhardt 2015-09-22 22:00:00 Bundesliga          0.110539  
                           2015-11-21 18:30:00 Bundesliga          0.185229  
                           2016-02-07 18:30:00 Bundesliga          0.181097  
                           2016-03-05 18:30:00 Bundesliga          0.224042  
                           2016-04-17 19:30:00 Bundesliga          0.245536

In [9]:
# Create datasets
blocks_per_input = 5

# Use differencing?
if differencing:
    train_dataset = DifferencedFootballDataset(train_df, blocks_per_input, multiple_players=True)
    test_dataset = DifferencedFootballDataset(test_df, blocks_per_input, multiple_players=True)
else:
    train_dataset = CustomFootballDataset(train_df, blocks_per_input, multiple_players=True)
    test_dataset = CustomFootballDataset(test_df, blocks_per_input, multiple_players=True)

In [10]:
len(train_dataset)

19924

In [11]:
# Create dataloaders

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [12]:
# Tuning params
params = {
    "learning_rates": [0.001],      
    "epochs": [20],        
    "layers": [1, 2],                            
    "h_sizes": [32, 64, 128, 256],                  
    "dropouts": [0.3]                
}

opt_hyper_params = hyperparam_tuning(params, stats_df=stats_df, train_dataloader=train_dataloader, 
                                             test_dataloader=test_dataloader, differenced=differencing)

print(f"""
      Best Hyperparameter setup \n
      - learning rate: {opt_hyper_params['learning_rate']} 
      - number of epochs: {opt_hyper_params['epoch']}.
      - number of layers: {opt_hyper_params['layers']}.
      - hidden size: {opt_hyper_params['hidden_size']}.
      - dropout: {opt_hyper_params['dropout']}.
      """)

learning_rates:   0%|          | 0/1 [00:00<?, ?it/s]

epochs:   0%|          | 0/1 [00:00<?, ?it/s]

layers:   0%|          | 0/2 [00:00<?, ?it/s]

h_sizes:   0%|          | 0/4 [00:00<?, ?it/s]

dropouts:   0%|          | 0/1 [00:00<?, ?it/s]

...

dropouts:   0%|          | 0/1 [00:00<?, ?it/s]

dropouts:   0%|          | 0/1 [00:00<?, ?it/s]

dropouts:   0%|          | 0/1 [00:00<?, ?it/s]

dropouts:   0%|          | 0/1 [00:00<?, ?it/s]


      Best Hyperparameter setup 

      - learning rate: 0.001 
      - number of epochs: 20.
      - number of layers: 2.
      - hidden size: 256.
      - dropout: 0.3.
      


In [13]:
# Tuned setup to MSE Loss and Adam
loss_fn = nn.MSELoss()
if differencing:
    model = DifferencingFootballLSTM(n_features=len(stats_df.columns), hidden_size=opt_hyper_params['hidden_size'], 
                     num_layers=opt_hyper_params['layers'], dropout=opt_hyper_params['dropout']).to(device)
else:
    model = FootballLSTM(n_features=len(stats_df.columns), hidden_size=opt_hyper_params['hidden_size'], 
                     num_layers=opt_hyper_params['layers'], dropout=opt_hyper_params['dropout']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=opt_hyper_params['learning_rate'])
num_epochs = opt_hyper_params['epoch']

# Train model
train_losses, test_losses = model.train_model(
    optimizer=optimizer,
    loss_fn=loss_fn,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    n_epochs=num_epochs,
    test_every=float('inf')
)

In [14]:
# Evaluate test performance

rmse, mae = model.eval_model(test_dataloader)
print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mae}")

Test RMSE: 0.3471454083919525
Test MAE: 0.2463754266500473


In [15]:
# See how model does on forecasting a specific player

player_df = stats_df.loc[150]

model.eval_model_on_player(player_df)

KeyError: 150

In [16]:
# Get predictions and actuals for each test player and save to csv

# For each test player, get most first blocks_per_input games, then predict the rest
# with LSTM. Also get approx. future dates, and use whatever league they last
# played in.

ids = []
names = []
dates = []
leagues = []
preds = []

max_look_ahead = 5 # Predict no more than 5 ahead for players

def get_k_future_dates(player_df, k):
    # Get the dates of existing blocks
    block_dates = player_df.index.get_level_values('date')
    
    # Calculate average time between consecutive blocks
    deltas = pd.Series(block_dates).diff().dropna()
    avg_delta = deltas.mean()
    
    # Project forward from last known date
    last_date = block_dates[-1]
    future_dates = [last_date + avg_delta * i for i in range(1, k + 1)]
    
    return future_dates

for (id, name), player_df in test_df.groupby(["player_id", "player_name"]):
    vals = player_df.values
    
    if differencing:
        if len(vals) > blocks_per_input + 1:  # +1 since differencing loses one row
            # Difference the series
            diff_vals = np.diff(vals, axis=0)  # (n_blocks - 1, n_features)
        
            x = torch.tensor(diff_vals[:blocks_per_input], dtype=torch.float32).unsqueeze(0)
        
            # Last known absolute value for undifferencing
            last_known = torch.tensor(vals[blocks_per_input], dtype=torch.float32).unsqueeze(0)
        
            remaining_games = len(vals) - blocks_per_input - 1  # -1 since last_known uses one block
            remaining_games = min(remaining_games, max_look_ahead)
        
            # predict_next_k now takes last_known and returns undifferenced predictions
            player_preds = model.predict_next_k(x, k=remaining_games, last_known=last_known).squeeze(0).detach().cpu().numpy()
        
            preds.append(player_preds)
            ids.extend([id] * remaining_games)
            names.extend([name] * remaining_games)
            dates.extend(get_k_future_dates(player_df.iloc[:blocks_per_input], remaining_games))
            leagues.extend([player_df.index.get_level_values('league')[blocks_per_input]] * remaining_games)
    else:
        if len(vals) > blocks_per_input:
            x = torch.tensor(vals[:blocks_per_input], dtype=torch.float32).unsqueeze(0)
            remaining_games = len(vals) - blocks_per_input
            remaining_games = min(remaining_games, max_look_ahead)
            player_preds = model.predict_next_k(x, k=remaining_games).squeeze(0).detach().cpu().numpy()
            preds.append(player_preds)
            ids.extend([id] * remaining_games)
            names.extend([name] * remaining_games)
            dates.extend(get_k_future_dates(player_df.iloc[:blocks_per_input], remaining_games))
            leagues.extend([player_df.index.get_level_values('league')[blocks_per_input]] * remaining_games) 
        
        
preds = np.concatenate(preds)

In [17]:
# Turn into a df
preds_df = pd.DataFrame(preds)
# Add id and name columns
preds_df.insert(0, 'player_id', ids)
preds_df.insert(1, 'player_name', names)
preds_df.insert(2, 'date', dates)
preds_df.insert(3, 'league', leagues)

# Rename other columns
preds_df = preds_df.rename(columns={i: f"{stats[i]}_per_90" for i in range(len(stats))})

preds_df.head()

,player_id,player_name,date,league,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90
0,27,Vladimir Darida,2015-05-22 19:22:30,Bundesliga,0.227607,0.265207,0.125019,0.129745,1.052266,0.460486,0.162312
1,27,Vladimir Darida,2015-07-10 00:15:00,Bundesliga,0.241826,0.276226,0.126531,0.131437,1.058103,0.478293,0.164957
2,27,Vladimir Darida,2015-08-27 05:07:30,Bundesliga,0.223879,0.259882,0.126799,0.132875,1.084764,0.471502,0.176261
3,27,Vladimir Darida,2015-10-14 10:00:00,Bundesliga,0.217011,0.250447,0.130181,0.139533,1.162124,0.469237,0.174716
4,27,Vladimir Darida,2015-12-01 14:52:30,Bundesliga,0.225327,0.251899,0.135847,0.144919,1.196297,0.482524,0.184469


In [18]:
# Pull transfer value data and merge
preds_df_combined = merge_stats_df_with_transfermarkt(preds_df, False)

In [19]:
# Add age and year (fractional columns
preds_df_combined["age"] = (preds_df_combined["date"] - preds_df_combined["date_of_birth"]).dt.days  / 365.25 # .25 accounts for leap years

preds_df_combined["year"] = preds_df_combined["date"].dt.year + (preds_df_combined["date"].dt.day_of_year / 365.25)

preds_df_combined.head()

,player_id,player_name,date,date_of_birth,league,goals_per_90,xG_per_90,assists_per_90,xA_per_90,key_passes_per_90,xGChain_per_90,xGBuildup_per_90,value,age,year
13229,3295.0,Javier Pastore,2015-04-30 10:00:00,1989-06-20,Ligue_1,0.210496,0.184026,0.190222,0.156963,1.526110,0.851624,0.580042,25000000,25.859001,2015.328542
13351,734.0,Wahbi Khazri,2015-05-07 19:45:00,1991-02-08,Ligue_1,0.298907,0.317290,0.164891,0.183556,1.563383,0.505963,0.171973,7000000,24.240931,2015.347707
13519,553.0,Anthony Martial,2015-04-26 02:22:30,1995-12-05,Ligue_1,0.309201,0.380814,0.125195,0.118138,0.867791,0.535012,0.139480,8000000,19.389459,2015.317591
14884,1611.0,Franco Brienza,2015-06-20 09:11:15,1979-03-19,Serie_A,0.188583,0.205777,0.160911,0.181518,1.635185,0.406042,0.175911,300000,36.254620,2015.468172
14921,3856.0,Milan Djuric,2015-06-19 00:33:45,1990-05-22,Serie_A,0.202096,0.242982,0.127828,0.128177,1.072785,0.434863,0.149984,500000,25.075975,2015.465435


In [20]:
# Get rid of DOB column
preds_df_combined = preds_df_combined.drop("date_of_birth", axis=1)

In [21]:
# Save to csv
preds_df_combined.to_csv(f"../data/{'_'.join(positions)}_predictions_real_values.csv", index=False)